# Agreement Between Embedding and Fingerprint Similarities

## Setup

In [ ]:
from setup import *

from torch.utils.data import random_split

## Compute Fingerprint Correlations

In [ ]:
def compute_fingerprint_correlations(sample_indices, embedding_cache, fingerprints):
    embeddings = embedding_cache["embeddings"].cpu()
    if not torch.equal(embedding_cache["id"], fingerprints["id"]): embeddings = embeddings[match_molecule_ids(embedding_cache["id"], fingerprints["id"])]
    similarities = embeddings[sample_indices] @ embeddings.T
    correlations = []
    for fp_type in ("ecfp", "rdkit"):
        fingerprint_similarities = pairwise_jaccard_similarity(fingerprints[fp_type][sample_indices], fingerprints[fp_type], device=DEVICE)
        correlations.append(spearman_correlations(similarities, fingerprint_similarities, min_pairs=2, nan_policy="propagate").tolist())
        del fingerprint_similarities
    return tuple(correlations)

embedding_cache = load_or_compute_embeddings(EMBEDDING_DIR / "embeddings_ANT.pt", CHECKPOINTS["ANT"], CHEMBL_DIR, DEVICE)
fingerprints = load_or_compute_fingerprints(FINGERPRINT_DIR, CHEMBL_DIR, SMILES_DIR, ids=embedding_cache["id"])
sample_size = 1000
_, val_split = random_split(range(len(embedding_cache["id"])), [0.95, 0.05], generator=torch.Generator().manual_seed(0))
sample_indices = np.sort(np.random.default_rng(0).choice(val_split.indices, size=min(sample_size, len(val_split)), replace=False))
del val_split, _
sample_smiles = [fingerprints["smiles"][index] for index in sample_indices]
fingerprint_correlations_ANT = compute_fingerprint_correlations(sample_indices, embedding_cache, fingerprints)
embedding_cache = load_or_compute_embeddings(EMBEDDING_DIR / "embeddings_NT.pt", CHECKPOINTS["NT"], CHEMBL_DIR, DEVICE)
fingerprint_correlations_NT = compute_fingerprint_correlations(sample_indices, embedding_cache, fingerprints)
embedding_cache = load_or_compute_embeddings(EMBEDDING_DIR / "embeddings_nopre.pt", CHECKPOINTS["nopre"], CHEMBL_DIR, DEVICE)
fingerprint_correlations_nopre = compute_fingerprint_correlations(sample_indices, embedding_cache, fingerprints)

ecfp_spearman_ANT, rdkit_spearman_ANT = fingerprint_correlations_ANT
ecfp_spearman_NT, rdkit_spearman_NT = fingerprint_correlations_NT
ecfp_spearman_nopre, rdkit_spearman_nopre = fingerprint_correlations_nopre

spearman_distributions = {
    ("ANT", "ECFP"): ecfp_spearman_ANT,
    ("ANT", "RDKFP"): rdkit_spearman_ANT,
    ("NT", "ECFP"): ecfp_spearman_NT,
    ("NT", "RDKFP"): rdkit_spearman_NT,
    ("nopre", "ECFP"): ecfp_spearman_nopre,
    ("nopre", "RDKFP"): rdkit_spearman_nopre,
}
sample_quantiles = (0.05, 0.25, 0.50, 0.75, 0.95)  # where along each distribution to pull a molecule from
fingerprint_stats = {key: summarize_distribution(values) for key, values in spearman_distributions.items()}
fingerprint_samples = {}
for key, values in spearman_distributions.items():
    arr = np.asarray(values, dtype=float)
    valid_positions = np.flatnonzero(~np.isnan(arr))
    sorted_positions = valid_positions[np.argsort(arr[valid_positions])]
    ranks = np.round(np.asarray(sample_quantiles) * (len(sorted_positions) - 1)).astype(int)
    fingerprint_samples[key] = [
        {"quantile": q, "position": int(sorted_positions[rank]),
         "dataset_idx": int(sample_indices[sorted_positions[rank]]),
         "smiles": sample_smiles[sorted_positions[rank]],
         "spearman": arr[sorted_positions[rank]]}
        for q, rank in zip(sample_quantiles, ranks)
    ]

## Figure 3: Fingerprint Correlation Distributions

In [ ]:
def plot_fingerprint_correlations(ax, ecfp_spearman, rdkit_spearman, bin_edges, title, yticks=np.arange(0, 0.061, 0.02)):
    ecfp_arr = np.asarray(ecfp_spearman)
    rdkit_arr = np.asarray(rdkit_spearman)
    ecfp_weights = np.ones_like(ecfp_arr) / len(ecfp_arr)
    rdkit_weights = np.ones_like(rdkit_arr) / len(rdkit_arr)
    ax.hist(ecfp_arr, bins=bin_edges, color="#9e0142", alpha=0.9, label="ECFP", weights=ecfp_weights)
    ax.hist(rdkit_arr, bins=bin_edges, color="#3288bd", alpha=0.9, label="RDKFP", weights=rdkit_weights)

    # compute actual bar heights (fraction of total per bin) to find the real max
    ecfp_counts, _ = np.histogram(ecfp_arr, bins=bin_edges, weights=ecfp_weights)
    rdkit_counts, _ = np.histogram(rdkit_arr, bins=bin_edges, weights=rdkit_weights)
    max_height = max(ecfp_counts.max(), rdkit_counts.max())
    ax.set_title(title, fontsize=6, pad=0)
    ax.set_xlim(-0.2, 0.8)
    ax.set_ylim(0.0, max_height * 1.1)  # small headroom so the tallest bar isn't flush with the top
    ax.set_yticks(yticks)
    ax.tick_params(labelsize=4, length=2, width=0.4, pad=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(0.4)

bin_width = 0.01
all_spearman = ecfp_spearman_ANT + rdkit_spearman_ANT + ecfp_spearman_NT + rdkit_spearman_NT + ecfp_spearman_nopre + rdkit_spearman_nopre
left_edge = np.floor(min(all_spearman) / bin_width) * bin_width
right_edge = np.ceil(max(all_spearman) / bin_width) * bin_width
bin_edges = np.arange(left_edge, right_edge + bin_width, bin_width)
fig, axes = plt.subplots(3, 1, figsize=(7.1 * cm, 7.3 * cm), sharex=True)
plot_fingerprint_correlations(axes[0], ecfp_spearman_ANT, rdkit_spearman_ANT, bin_edges, MODEL_TITLES["ANT"])
axes[0].legend(frameon=False, fontsize=5, markerscale=0.5, loc="upper left", handlelength=1)
axes[0].tick_params(axis="x", which="both", bottom=True, labelbottom=False)
plot_fingerprint_correlations(axes[1], ecfp_spearman_NT, rdkit_spearman_NT, bin_edges, MODEL_TITLES["NT"], yticks=np.arange(0, 0.31, 0.1))

plot_fingerprint_correlations(axes[2], ecfp_spearman_nopre, rdkit_spearman_nopre, bin_edges, MODEL_TITLES["nopre"], yticks=np.arange(0, 0.081, 0.02))
fig.text(0.02, 0.5, "Fraction of Total", fontsize=6, rotation=90, ha="center", va="center")
axes[2].set_xlabel("Spearman Correlation (FP vs. Embedding Cosine Sim.)", fontsize=6, ha="center", va="center")

fig.subplots_adjust(hspace=0.15, left=0.09, top=0.97, bottom=0.08, right=0.98) 
fig.savefig(OUTPUT_DIR / "spearman_correlation_histogram.png", dpi=600)
plt.show()

## Fingerprint Correlation Summary

In [ ]:
display(pd.DataFrame([
    {"Model": model, "Fingerprint": fingerprint, **statistics}
    for (model, fingerprint), statistics in fingerprint_stats.items()
]).round(4))

## Representative Query Molecules

In [ ]:
display(pd.DataFrame([
    {"Model": model, "Fingerprint": fingerprint, **sample}
    for (model, fingerprint), samples in fingerprint_samples.items()
    for sample in samples
]))